### Document Ingestion

In [1]:
#Document Structure

from langchain_core.documents import Document

In [2]:
doc = Document(
    page_content = "this is the main text content I am using to create RAG",
    metadata = {
        "source":"example.txt",
        "pages":1,
        "author":"Anusha Jain",
        "date_created":"2026-11-14"
    }
)

doc

Document(metadata={'source': 'example.txt', 'pages': 1, 'author': 'Anusha Jain', 'date_created': '2026-11-14'}, page_content='this is the main text content I am using to create RAG')

In [4]:
### PDF Loader
from langchain_community.document_loaders import PyPDFLoader

loader = PyPDFLoader("../data/pdf/chemistry_notes.pdf")
loader.load()

C:\Users\Abheek Jain\AppData\Local\Temp\ipykernel_22364\3857141098.py:2: DeprecationWarning: `langchain-community` is being sunset and is no longer actively maintained. See https://github.com/langchain-ai/langchain-community/issues/674 for details and migration guidance toward standalone integration packages.
  from langchain_community.document_loaders import PyPDFLoader


[Document(metadata={'producer': 'Microsoft® PowerPoint® 2013', 'creator': 'Microsoft® PowerPoint® 2013', 'creationdate': '2022-10-12T08:35:47+05:30', 'title': 'PowerPoint Presentation', 'author': 'Kalisadhan Mukherjee', 'moddate': '2022-10-12T08:35:47+05:30', 'source': '../data/pdf/chemistry_notes.pdf', 'total_pages': 104, 'page': 0, 'page_label': '1'}, page_content='Chemistry of Engineering Materials\nReference book: Wiley Engineering Chemistry\nMaterials Science and Engineering-An Introduction; Callister\nUNIT 1: Chemistry of Engineering Materials 10 h \nTraditional Materials: Introduction and classification of materials; metallic materials,\npolymeric, ceramic materials\nAdvanced Materials: Introduction to nanomaterials: Properties and application;\nCarbonaceous materials (fullerene, carbon nanotube, graphene, etc.); Composite\nmaterials; Liquid crystals: Classification and Application'),
 Document(metadata={'producer': 'Microsoft® PowerPoint® 2013', 'creator': 'Microsoft® PowerPoin

In [5]:
### Directory Loader
from langchain_community.document_loaders import DirectoryLoader

#load all pdf files from the directory
dir_loader = DirectoryLoader(
    "../data/pdf",
    glob="**/*.pdf", ##Pattern to match
    loader_cls = PyPDFLoader, ##Loader class we want to use
)

dir_loader

In [8]:
documents = dir_loader.load()
documents

[Document(metadata={'producer': 'Microsoft® PowerPoint® 2013', 'creator': 'Microsoft® PowerPoint® 2013', 'creationdate': '2022-10-12T08:35:47+05:30', 'title': 'PowerPoint Presentation', 'author': 'Kalisadhan Mukherjee', 'moddate': '2022-10-12T08:35:47+05:30', 'source': '..\\data\\pdf\\chemistry_notes.pdf', 'total_pages': 104, 'page': 0, 'page_label': '1'}, page_content='Chemistry of Engineering Materials\nReference book: Wiley Engineering Chemistry\nMaterials Science and Engineering-An Introduction; Callister\nUNIT 1: Chemistry of Engineering Materials 10 h \nTraditional Materials: Introduction and classification of materials; metallic materials,\npolymeric, ceramic materials\nAdvanced Materials: Introduction to nanomaterials: Properties and application;\nCarbonaceous materials (fullerene, carbon nanotube, graphene, etc.); Composite\nmaterials; Liquid crystals: Classification and Application'),
 Document(metadata={'producer': 'Microsoft® PowerPoint® 2013', 'creator': 'Microsoft® PowerP

In [9]:
from langchain_text_splitters import RecursiveCharacterTextSplitter
from pathlib import Path
### Read all the pdf's inside the directory
def process_all_pdfs(pdf_directory):
    """Process all PDF files in a directory"""
    all_documents = []
    pdf_dir = Path(pdf_directory)
    
    # Find all PDF files recursively
    pdf_files = list(pdf_dir.glob("**/*.pdf"))
    
    print(f"Found {len(pdf_files)} PDF files to process")
    
    for pdf_file in pdf_files:
        print(f"\nProcessing: {pdf_file.name}")
        try:
            loader = PyPDFLoader(str(pdf_file))
            documents = loader.load()
            
            # Add source information to metadata
            for doc in documents:
                doc.metadata['source_file'] = pdf_file.name
                doc.metadata['file_type'] = 'pdf'
            
            all_documents.extend(documents)
            print(f"  ✓ Loaded {len(documents)} pages")
            
        except Exception as e:
            print(f"  ✗ Error: {e}")
    
    print(f"\nTotal documents loaded: {len(all_documents)}")
    return all_documents

# Process all PDFs in the data directory
all_pdf_documents = process_all_pdfs("../data")

Found 2 PDF files to process

Processing: chemistry_notes.pdf
  ✓ Loaded 104 pages

Processing: eeee_notes.pdf
  ✓ Loaded 688 pages

Total documents loaded: 792


In [10]:
all_pdf_documents

[Document(metadata={'producer': 'Microsoft® PowerPoint® 2013', 'creator': 'Microsoft® PowerPoint® 2013', 'creationdate': '2022-10-12T08:35:47+05:30', 'title': 'PowerPoint Presentation', 'author': 'Kalisadhan Mukherjee', 'moddate': '2022-10-12T08:35:47+05:30', 'source': '..\\data\\pdf\\chemistry_notes.pdf', 'total_pages': 104, 'page': 0, 'page_label': '1', 'source_file': 'chemistry_notes.pdf', 'file_type': 'pdf'}, page_content='Chemistry of Engineering Materials\nReference book: Wiley Engineering Chemistry\nMaterials Science and Engineering-An Introduction; Callister\nUNIT 1: Chemistry of Engineering Materials 10 h \nTraditional Materials: Introduction and classification of materials; metallic materials,\npolymeric, ceramic materials\nAdvanced Materials: Introduction to nanomaterials: Properties and application;\nCarbonaceous materials (fullerene, carbon nanotube, graphene, etc.); Composite\nmaterials; Liquid crystals: Classification and Application'),
 Document(metadata={'producer': 'M

In [11]:
### Text splitting get into chunks

def split_documents(documents,chunk_size=1000,chunk_overlap=200):
    """Split documents into smaller chunks for better RAG performance"""
    text_splitter = RecursiveCharacterTextSplitter(
        chunk_size=chunk_size,
        chunk_overlap=chunk_overlap,
        length_function=len,
        separators=["\n\n", "\n", " ", ""]
    )
    split_docs = text_splitter.split_documents(documents)
    print(f"Split {len(documents)} documents into {len(split_docs)} chunks")
    
    # Show example of a chunk
    if split_docs:
        print(f"\nExample chunk:")
        print(f"Content: {split_docs[0].page_content[:200]}...")
        print(f"Metadata: {split_docs[0].metadata}")
    
    return split_docs


In [12]:
chunks=split_documents(all_pdf_documents)
chunks

Split 792 documents into 1070 chunks

Example chunk:
Content: Chemistry of Engineering Materials
Reference book: Wiley Engineering Chemistry
Materials Science and Engineering-An Introduction; Callister
UNIT 1: Chemistry of Engineering Materials 10 h 
Traditional...
Metadata: {'producer': 'Microsoft® PowerPoint® 2013', 'creator': 'Microsoft® PowerPoint® 2013', 'creationdate': '2022-10-12T08:35:47+05:30', 'title': 'PowerPoint Presentation', 'author': 'Kalisadhan Mukherjee', 'moddate': '2022-10-12T08:35:47+05:30', 'source': '..\\data\\pdf\\chemistry_notes.pdf', 'total_pages': 104, 'page': 0, 'page_label': '1', 'source_file': 'chemistry_notes.pdf', 'file_type': 'pdf'}


[Document(metadata={'producer': 'Microsoft® PowerPoint® 2013', 'creator': 'Microsoft® PowerPoint® 2013', 'creationdate': '2022-10-12T08:35:47+05:30', 'title': 'PowerPoint Presentation', 'author': 'Kalisadhan Mukherjee', 'moddate': '2022-10-12T08:35:47+05:30', 'source': '..\\data\\pdf\\chemistry_notes.pdf', 'total_pages': 104, 'page': 0, 'page_label': '1', 'source_file': 'chemistry_notes.pdf', 'file_type': 'pdf'}, page_content='Chemistry of Engineering Materials\nReference book: Wiley Engineering Chemistry\nMaterials Science and Engineering-An Introduction; Callister\nUNIT 1: Chemistry of Engineering Materials 10 h \nTraditional Materials: Introduction and classification of materials; metallic materials,\npolymeric, ceramic materials\nAdvanced Materials: Introduction to nanomaterials: Properties and application;\nCarbonaceous materials (fullerene, carbon nanotube, graphene, etc.); Composite\nmaterials; Liquid crystals: Classification and Application'),
 Document(metadata={'producer': 'M

### Embeddings and Vector DB


In [13]:
import numpy as np
from sentence_transformers import SentenceTransformer
import chromadb
from chromadb.config import Settings
import uuid
from typing import List, Tuple, Any, Dict
from sklearn.metrics.pairwise import cosine_similarity

c:\Users\Abheek Jain\Documents\ai-ml-engineer\python basics\day_08_rag\.venv\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [15]:
class EmbeddingManager:
    """Handles document embedding with SentenceTransformer"""

    def __init__(self, model_name: str="all-MiniLM-L6-v2"):
        """
        Initialize the embedding manager

        Args: model name foe sentence embeddings - taken from hugging face. Dimension is 384 
        """

        self.model_name = model_name
        self.model = None
        self._load_model()

    def _load_model(self):
        "Load the sentence transformer model"

        try:
            print(f"Loading embedding model: {self.model_name}")
            self.model = SentenceTransformer(self.model_name)
            print(f"Model loaded successfully. Embedding dimension :{self.model.get_embedding_dimension}")
        except Exception as e:
            print(f"Error loading model {self.model_name}: {e}")
            raise

    def generate_embeddings(self, texts: List[str]) -> np.ndarray:
        """
        Generate embeddings for a list of texts to embed

        Args:
            texts: List of strings that need to be embedded 

        Returns:
            numpy array with embeddings with shape (len(texts), embedding_dim)
        """

        if not self.model:
            raise ValueError("Model not loaded")

        print(f"Generating embeddings for {len(texts)} texts ...")
        embeddings = self.model.encode(texts)
        print(f"Generated embeddings with shape: {embeddings.shape}")
        return embeddings

embedding_manager = EmbeddingManager()
embedding_manager

Loading embedding model: all-MiniLM-L6-v2


Loading weights: 100%|██████████| 103/103 [00:00<00:00, 1198.39it/s]


Model loaded successfully. Embedding dimension :<bound method SentenceTransformer.get_embedding_dimension of SentenceTransformer(
  (0): Transformer({'transformer_task': 'feature-extraction', 'modality_config': {'text': {'method': 'forward', 'method_output_name': 'last_hidden_state'}}, 'module_output_name': 'token_embeddings', 'architecture': 'BertModel'})
  (1): Pooling({'embedding_dimension': 384, 'pooling_mode': 'mean', 'include_prompt': True})
  (2): Normalize({'module_input_name': 'sentence_embedding', 'module_output_name': 'sentence_embedding'})
)>


### Vector Store

In [21]:
class VectorStore:
    """Manages document embeddings in a ChromaDB vector store"""
    
    def __init__(self, collection_name: str = "pdf_documents", persist_directory: str = "../data/vector_store"):
        """
        Initialize the vector store
        
        Args:
            collection_name: Name of the ChromaDB collection
            persist_directory: Directory to persist the vector store
        """
        self.collection_name = collection_name
        self.persist_directory = persist_directory
        self.client = None
        self.collection = None
        self._initialize_store()

    def _initialize_store(self):
        """Initialize ChromaDB client and collection"""
        try:
            # Create persistent ChromaDB client
            os.makedirs(self.persist_directory, exist_ok=True)
            self.client = chromadb.PersistentClient(path=self.persist_directory)
            
            # Get or create collection
            self.collection = self.client.get_or_create_collection(
                name=self.collection_name,
                metadata={"description": "PDF document embeddings for RAG"}
            )
            print(f"Vector store initialized. Collection: {self.collection_name}")
            print(f"Existing documents in collection: {self.collection.count()}")
            
        except Exception as e:
            print(f"Error initializing vector store: {e}")
            raise

    def add_documents(self, documents: List[Any], embeddings: np.ndarray):
        """
        Add documents and their embeddings to the vector store
        
        Args:
            documents: List of LangChain documents
            embeddings: Corresponding embeddings for the documents
        """
        if len(documents) != len(embeddings):
            raise ValueError("Number of documents must match number of embeddings")
        
        print(f"Adding {len(documents)} documents to vector store...")
        
        # Prepare data for ChromaDB
        ids = []
        metadatas = []
        documents_text = []
        embeddings_list = []
        
        for i, (doc, embedding) in enumerate(zip(documents, embeddings)):
            # Generate unique ID
            doc_id = f"doc_{uuid.uuid4().hex[:8]}_{i}"
            ids.append(doc_id)
            
            # Prepare metadata
            metadata = dict(doc.metadata)
            metadata['doc_index'] = i
            metadata['content_length'] = len(doc.page_content)
            metadatas.append(metadata)
            
            # Document content
            documents_text.append(doc.page_content)
            
            # Embedding
            embeddings_list.append(embedding.tolist())
        
        # Add to collection
        try:
            self.collection.add(
                ids=ids,
                embeddings=embeddings_list,
                metadatas=metadatas,
                documents=documents_text
            )
            print(f"Successfully added {len(documents)} documents to vector store")
            print(f"Total documents in collection: {self.collection.count()}")
            
        except Exception as e:
            print(f"Error adding documents to vector store: {e}")
            raise

vectorstore=VectorStore()
vectorstore
    

Vector store initialized. Collection: pdf_documents
Existing documents in collection: 0


In [22]:
chunks

[Document(metadata={'producer': 'Microsoft® PowerPoint® 2013', 'creator': 'Microsoft® PowerPoint® 2013', 'creationdate': '2022-10-12T08:35:47+05:30', 'title': 'PowerPoint Presentation', 'author': 'Kalisadhan Mukherjee', 'moddate': '2022-10-12T08:35:47+05:30', 'source': '..\\data\\pdf\\chemistry_notes.pdf', 'total_pages': 104, 'page': 0, 'page_label': '1', 'source_file': 'chemistry_notes.pdf', 'file_type': 'pdf'}, page_content='Chemistry of Engineering Materials\nReference book: Wiley Engineering Chemistry\nMaterials Science and Engineering-An Introduction; Callister\nUNIT 1: Chemistry of Engineering Materials 10 h \nTraditional Materials: Introduction and classification of materials; metallic materials,\npolymeric, ceramic materials\nAdvanced Materials: Introduction to nanomaterials: Properties and application;\nCarbonaceous materials (fullerene, carbon nanotube, graphene, etc.); Composite\nmaterials; Liquid crystals: Classification and Application'),
 Document(metadata={'producer': 'M

In [23]:
### Convert the text to embeddings
texts=[doc.page_content for doc in chunks]

## Generate the Embeddings

embeddings=embedding_manager.generate_embeddings(texts)

##store int he vector dtaabase
vectorstore.add_documents(chunks,embeddings)

Generating embeddings for 1070 texts ...
Generated embeddings with shape: (1070, 384)
Adding 1070 documents to vector store...
Successfully added 1070 documents to vector store
Total documents in collection: 1070


### Retriever Pipeline From VectorStore

In [24]:
class RAGRetriever:
    """Handles query-based retrieval from the vector store"""
    
    def __init__(self, vector_store: VectorStore, embedding_manager: EmbeddingManager):
        """
        Initialize the retriever
        
        Args:
            vector_store: Vector store containing document embeddings
            embedding_manager: Manager for generating query embeddings
        """
        self.vector_store = vector_store
        self.embedding_manager = embedding_manager

    def retrieve(self, query: str, top_k: int = 5, score_threshold: float = 0.0) -> List[Dict[str, Any]]:
        """
        Retrieve relevant documents for a query
        
        Args:
            query: The search query
            top_k: Number of top results to return
            score_threshold: Minimum similarity score threshold
            
        Returns:
            List of dictionaries containing retrieved documents and metadata
        """
        print(f"Retrieving documents for query: '{query}'")
        print(f"Top K: {top_k}, Score threshold: {score_threshold}")
        
        # Generate query embedding
        query_embedding = self.embedding_manager.generate_embeddings([query])[0]
        
        # Search in vector store
        try:
            results = self.vector_store.collection.query(
                query_embeddings=[query_embedding.tolist()],
                n_results=top_k
            )
            
            # Process results
            retrieved_docs = []
            
            if results['documents'] and results['documents'][0]:
                documents = results['documents'][0]
                metadatas = results['metadatas'][0]
                distances = results['distances'][0]
                ids = results['ids'][0]
                
                for i, (doc_id, document, metadata, distance) in enumerate(zip(ids, documents, metadatas, distances)):
                    # Convert distance to similarity score (ChromaDB uses cosine distance)
                    similarity_score = 1 - distance
                    
                    if similarity_score >= score_threshold:
                        retrieved_docs.append({
                            'id': doc_id,
                            'content': document,
                            'metadata': metadata,
                            'similarity_score': similarity_score,
                            'distance': distance,
                            'rank': i + 1
                        })
                
                print(f"Retrieved {len(retrieved_docs)} documents (after filtering)")
            else:
                print("No documents found")
            
            return retrieved_docs
            
        except Exception as e:
            print(f"Error during retrieval: {e}")
            return []

rag_retriever=RAGRetriever(vectorstore,embedding_manager)



In [25]:
rag_retriever

In [27]:
rag_retriever.retrieve("What is photonic devices")

Retrieving documents for query: 'What is photonic devices'
Top K: 5, Score threshold: 0.0
Generating embeddings for 1 texts ...
Generated embeddings with shape: (1, 384)
Retrieved 2 documents (after filtering)


[{'id': 'doc_35cdf3b2_11',
  'content': '• Photonic materials include\n\uf076 a number of compound semiconductors designed for light emission\nor detection;\n\uf076 elemental dopants that serve as photonic performance-control\nagents;\n\uf076 metal- or diamond-film heat sinks;\n\uf076 metalized films for contacts, physical barriers, and bonding;\n\uf076 and silica glass, ceramics, and rare earths for optical fibres.',
  'metadata': {'source': '..\\data\\pdf\\chemistry_notes.pdf',
   'doc_index': 11,
   'page_label': '11',
   'total_pages': 104,
   'page': 10,
   'moddate': '2022-10-12T08:35:47+05:30',
   'content_length': 346,
   'source_file': 'chemistry_notes.pdf',
   'title': 'PowerPoint Presentation',
   'file_type': 'pdf',
   'author': 'Kalisadhan Mukherjee',
   'creator': 'Microsoft® PowerPoint® 2013',
   'producer': 'Microsoft® PowerPoint® 2013',
   'creationdate': '2022-10-12T08:35:47+05:30'},
  'similarity_score': 0.2677851915359497,
  'distance': 0.7322148084640503,
  'rank':

In [38]:
rag_retriever.retrieve("What IS CVD")

Retrieving documents for query: 'What IS CVD'
Top K: 5, Score threshold: 0.0
Generating embeddings for 1 texts ...
Generated embeddings with shape: (1, 384)
Retrieved 1 documents (after filtering)


[{'id': 'doc_653be656_76',
  'content': 'What is CVD',
  'metadata': {'doc_index': 76,
   'creator': 'Microsoft® PowerPoint® 2013',
   'file_type': 'pdf',
   'page': 74,
   'title': 'PowerPoint Presentation',
   'author': 'Kalisadhan Mukherjee',
   'content_length': 11,
   'source': '..\\data\\pdf\\chemistry_notes.pdf',
   'page_label': '75',
   'moddate': '2022-10-12T08:35:47+05:30',
   'source_file': 'chemistry_notes.pdf',
   'producer': 'Microsoft® PowerPoint® 2013',
   'total_pages': 104,
   'creationdate': '2022-10-12T08:35:47+05:30'},
  'similarity_score': 0.9999999999997646,
  'distance': 2.3539780151168865e-13,
  'rank': 1}]

In [39]:
rag_retriever.retrieve("What ARE METALLIC MATERIALS")

Retrieving documents for query: 'What ARE METALLIC MATERIALS'
Top K: 5, Score threshold: 0.0
Generating embeddings for 1 texts ...
Generated embeddings with shape: (1, 384)
Retrieved 5 documents (after filtering)


[{'id': 'doc_6646a2ac_18',
  'content': 'Metallic materials are normally combination of one or more metallic elements and\nmay also contain some non-metallic elements like C, N, O, etc.\nMetals have a crystalline structure in which the atoms are arranged in an orderly\nmanner.\nThey are quite strong, yet deformable, which accounts for their extensive use in\nstructural applications.\nMetals are commonly divided into two classes namely, i) Ferrous Metals and ii) Non-\nFerrous Metals.\nMetals account for about two thirds of all the elements and about 24% of the mass of\nthe planet.\nMetals have useful properties including strength, ductility, high melting points,\nthermal and electrical conductivity, and toughness.\nMetallic Materials\n10/12/2022 Dr. Anu Manhas 18',
  'metadata': {'author': 'Kalisadhan Mukherjee',
   'total_pages': 104,
   'content_length': 718,
   'creationdate': '2022-10-12T08:35:47+05:30',
   'file_type': 'pdf',
   'page_label': '18',
   'doc_index': 18,
   'moddate':

In [40]:
rag_retriever.retrieve("What is star and delta connections")

Retrieving documents for query: 'What is star and delta connections'
Top K: 5, Score threshold: 0.0
Generating embeddings for 1 texts ...
Generated embeddings with shape: (1, 384)
Retrieved 5 documents (after filtering)


[{'id': 'doc_8c53ab76_732',
  'content': '5.14 Basic Electrical Engineering\nFor a delta-connected load,\n  Vph = VL\n Iph = \nV\nZ\nV\nZ\nph\nph\nL\nph\n=\n  IL = 3  Iph = 3 V Z L ph\n PD = 3  VL IL cos f\n  =  3  ¥ VL ¥ 3  V Z L ph  ¥ cos f\n  = 3 \nV\nZ\nL\nph\n2\n cos f \n  = 3PY\n PY = 1 3 PD\nThus, power consumed by a balanced star-connected load is one-third of that in the \ncase of a delta-connected load.\n5.12 COMPARISOn BeTWeen STAR  \nAnD DeLTA COnneCTIOnS\nStar Connection Delta Connection\n1. VL = 3 Vph 1. VL = Vph\n2. IL = Iph 2. IL = 3 Iph\n3. Line voltage leads the respective phase \nvoltage by 30°.\n3. Line current lags behind the respective \nphase current by 30°.\n4. Power in star connection is one-third of \npower in delta connection.\n4. Power in delta connection is 3 times of \nthe power in star connection.\n5. Three-phase, three-wire and three-phase, \nfour-wire systems are possible.\n5. Only three-phase, three-wire system is \npossible.\n6. The phasor sum of all 

In [42]:
rag_retriever.retrieve("limitations of ohm's law")

Retrieving documents for query: 'limitations of ohm's law'
Top K: 5, Score threshold: 0.0
Generating embeddings for 1 texts ...
Generated embeddings with shape: (1, 384)
Retrieved 1 documents (after filtering)


[{'id': 'doc_028e5cb6_125',
  'content': '1.5 Resistance 1.5\n1.4 Ohm’S Law\nAccording to Ohm’s law, the potential difference across any two points on a conductor is \ndirectly proportional to the current flowing through it, provided the physical conditions, viz., \nmaterial length, cross\xadsectional area and temperature of the conductor remain constant.\n V µ I\n V = R I\nwhere R is the resistance between two points of the conductor.\nLimitations\n 1. Ohm’s law does not apply to nonmetallic conductors. For example, for silicon \ncarbide, the relationship is given by V = KI m where K and m are constants and m is \nless than unity.\n 2. Ohm’s law also does not apply to nonlinear devices such as zener diodes, voltage \nregulator tubes, etc.\n 3. Ohm’s law is true for metal conductors at constant temperature. If the temperature \nchanges, the law is not applicable.\n1.5 ReSIStanCe\nResistance is the property of a material due to which it opposes the flow of electric \ncurrent through it.